# Does the single-axis-fixed irregular grid contradict the boustrophedon path or the return path?

`compare_60x_40x_irregular_grid_fov_coverage.ipynb` prototyped "Method 2"
(one axis regular/fixed, the other axis rebuilt independently per row/
column, snug to that row/column's own local tissue extent) and showed it
needs fewer FOVs than a plain regular grid -- but its own path ordering was
"a rough stand-in" (alternate direction per fixed-position index) and it
never touched production's real path-building pipeline
(`create_grid_positions` -> `generate_scanning_path` -> `close_scanning_path`).

Before promoting Method 2 into `acquisition/positions.py`, this notebook
checks it against that pipeline directly:

1. **Boustrophedon path**: build a proper generalisation of
   `generate_scanning_path` for variable-length rows/columns, and prove it
   is not just "similar" but **exactly reproduces** `generate_scanning_path`
   or a plain rectangular tissue (where the irregular grid degenerates to a
   regular one) -- a real regression check, not a visual comparison.
2. **Return path**: run `close_scanning_path` on the resulting irregular
   path and cross-check its quantised `_side_indices` selection against an
   independent, real-coordinate ground truth -- `close_scanning_path`
   assumes a shared global lattice on BOTH axes, but the irregular grid only
   has one (the fixed axis); this checks whether that assumption still
   holds for the pairing actually used (return side along the CROSS axis).
3. **Short-return-leg parity rule**: `create_grid_positions` forces its
   traversal axis to an EVEN count specifically so the snake's start/end
   share a side (short return leg) -- Method 2's original prototype never
   applied this to the fixed axis. Checked here with a real before/after
   comparison of the resulting return-leg length.
4. **Choosing `return_side` automatically**: rather than guess which side
   to close on, determine it from the raw path's own start/end geometry
   (falling back to an empirical shorter-`max_step_um` comparison when
   that's ambiguous) -- verified against a real, human-made wrong guess
   this notebook itself made in an earlier version (see the Takeaways).
5. Re-confirms the single-connected-component hard constraint on the
   final, properly-ordered + closed path (not just the raw unordered
   coordinate set).

Everything here is prototyped in this notebook only -- nothing is written
to `acquisition/positions.py` yet.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import box as shapely_box
from shapely.ops import unary_union
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components, minimum_spanning_tree

# notebooks/tests/<subfolder>/ is three levels under the repo root (MERci/),
# same convention as notebooks/before_imaging/regular/ (3 levels).
MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/
SAMPLE_DIR = MERCI_DIR.parent                  # this repo's own sandbox experiment
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs   import get_camera_pixel_size_um, get_camera_frame_size
from MERci.acquisition.positions import (
    load_boundary_polygon, load_hole_polygons,
    create_grid_positions, generate_scanning_path, filter_scanning_path,
    close_scanning_path, get_path_stats,
    spaced_coords, _grid_indices, _side_indices,   # production internals -- reused, not reimplemented
)
from MERci.acquisition.mosaic import load_mosaic_canvas
from MERci.visualization import get_merci_figures_dir

## Part 1 -- the irregular grid + boustrophedon path builder (prototype)

`build_irregular_bands` is the same per-row/column, hole-aware, local-piece
construction as the prior notebook's `build_irregular_grid`, but returns
**bands** (`(fixed_value, ascending cross-axis array)`, one per fixed-axis
lattice position, in ascending fixed-value order) instead of a flat point
list -- the shape `generate_irregular_scanning_path` needs below.

The fixed axis is built with production's own `spaced_coords` (imported,
not reimplemented) with `even=True` -- the exact parity `create_grid_positions`
forces on its own traversal axis, for the same short-return-leg reason (see
its docstring). Each band's own cross-axis piece(s) use `spaced_coords`
with `even=False` (one centred piece per contiguous strip of real tissue) --
identical to `create_grid_positions`' own cross-axis call when a band has
exactly one piece spanning the whole cross extent (the degenerate
rectangular case checked in Part 2).

In [ ]:
def build_irregular_bands(boundary_polygon, hole_polygons, step_size_um, fov_size_um,
                           fixed_axis="y", min_width_frac=0.1, fixed_offset=0.0,
                           force_parity=True):
    """Build per-band (fixed_axis regular-lattice) cross-axis position lists.

    Returns a list of (fixed_value, cross_array_ascending), one entry per
    fixed-axis lattice position, in ASCENDING fixed_value order -- the exact
    contract generate_irregular_scanning_path expects. A band with no real
    tissue in its strip is still included, with an empty cross array (kept
    so band INDEX still lines up with the fixed-axis lattice position for
    the alternation logic below; empty bands simply contribute no points).

    force_parity=True (default) forces the fixed axis to an EVEN count via
    spaced_coords -- create_grid_positions' own short-return-leg parity
    rule (see its docstring), applied here to the fixed axis since it is
    the one axis that stays on a single global lattice. force_parity=False
    (ablation only, see Part 4) uses the plain smallest-span-covering count
    instead, whatever parity that happens to produce.
    """
    tissue = boundary_polygon.difference(unary_union(hole_polygons)) if hole_polygons else boundary_polygon
    xmin, ymin, xmax, ymax = boundary_polygon.bounds
    half_h = fov_size_um / 2.0
    min_width_um = min_width_frac * fov_size_um

    if fixed_axis == "y":
        fixed_min, fixed_max, cross_min, cross_max = ymin, ymax, xmin, xmax
    elif fixed_axis == "x":
        fixed_min, fixed_max, cross_min, cross_max = xmin, xmax, ymin, ymax
    else:
        raise ValueError("fixed_axis must be 'x' or 'y'")

    fixed_center = (fixed_min + fixed_max) / 2.0 + fixed_offset
    if force_parity:
        fixed_positions = spaced_coords(fixed_center, fixed_min, fixed_max, step_size_um, even=True)
    else:
        span = fixed_max - fixed_min
        n = max(1, int(np.ceil(span / step_size_um)))
        fixed_positions = fixed_center + (np.arange(n) - (n - 1) / 2.0) * step_size_um

    bands = []
    for f in fixed_positions:
        lo_f, hi_f = f - half_h, f + half_h
        strip = (shapely_box(cross_min - 1.0, lo_f, cross_max + 1.0, hi_f) if fixed_axis == "y"
                 else shapely_box(lo_f, cross_min - 1.0, hi_f, cross_max + 1.0))
        inter = tissue.intersection(strip)
        cross_vals = []
        if not inter.is_empty:
            pieces = list(inter.geoms) if hasattr(inter, "geoms") else [inter]
            pieces.sort(key=lambda p: p.bounds[0] if fixed_axis == "y" else p.bounds[1])
            for piece in pieces:
                if piece.is_empty:
                    continue
                pxmin, pymin, pxmax, pymax = piece.bounds
                lo, hi = (pxmin, pxmax) if fixed_axis == "y" else (pymin, pymax)
                if (hi - lo) < min_width_um:
                    continue
                piece_positions = spaced_coords((lo + hi) / 2.0, lo, hi, step_size_um, even=False)
                cross_vals.extend(piece_positions.tolist())
        bands.append((float(f), np.array(sorted(cross_vals))))
    return bands


def generate_irregular_scanning_path(bands, fixed_axis):
    """Order per-band cross-axis positions into a boustrophedon path.

    Generalises generate_scanning_path (acquisition/positions.py) to
    variable-length bands instead of a fixed-width (H, W) grid -- but
    replicates that function's own traversal/alternation loop structure
    exactly (not a reformulation), so the two agree exactly whenever every
    band shares the same cross-axis positions (Part 2's regression check).
    *bands* must be in ASCENDING fixed-axis order, matching
    create_grid_positions' own ascending xs/ys and build_irregular_bands'
    contract above.
    """
    path = []
    n_bands = len(bands)
    if fixed_axis == "y":         # mirrors generate_scanning_path(direction="horizontal")
        for strip, i in enumerate(range(n_bands - 1, -1, -1)):
            fixed_val, cross_vals = bands[i]
            ordered = cross_vals if strip % 2 == 0 else cross_vals[::-1]
            for c in ordered:
                path.append((c, fixed_val))
    elif fixed_axis == "x":       # mirrors generate_scanning_path(direction="vertical")
        for j in range(n_bands):
            fixed_val, cross_vals = bands[j]
            ordered = cross_vals[::-1] if j % 2 == 0 else cross_vals
            for c in ordered:
                path.append((fixed_val, c))
    else:
        raise ValueError("fixed_axis must be 'x' or 'y'")
    return np.array(path) if path else np.empty((0, 2))


def build_irregular_boundary_path(boundary_polygon, hole_polygons, step_size_um, fov_size_um,
                                   fixed_axis="y", min_width_frac=0.1, fixed_offset=0.0,
                                   force_parity=True):
    """Full pipeline: build_irregular_bands -> generate_irregular_scanning_path
    -> filter_scanning_path -- the irregular-grid analogue of production's
    create_grid_positions -> generate_scanning_path -> filter_scanning_path.
    """
    bands = build_irregular_bands(boundary_polygon, hole_polygons, step_size_um, fov_size_um,
                                   fixed_axis=fixed_axis, min_width_frac=min_width_frac,
                                   fixed_offset=fixed_offset, force_parity=force_parity)
    path = generate_irregular_scanning_path(bands, fixed_axis=fixed_axis)
    if len(path) == 0:
        return path
    return filter_scanning_path(path, boundary_polygon, hole_polygons, fov_size_um)

## Part 2 -- regression check: does it exactly reproduce `generate_scanning_path`?

On a plain rectangular tissue (no holes), every band has exactly ONE piece
spanning the whole cross extent -- so `build_irregular_bands` should give
every band the SAME cross-axis lattice `create_grid_positions` itself would
build, and `generate_irregular_scanning_path` should then produce the
IDENTICAL ordered path `generate_scanning_path` does. This is checked as an
exact array equality (`np.allclose`), not eyeballed -- any divergence here
means the irregular-grid path genuinely contradicts the production
boustrophedon builder.

In [ ]:
STEP_TEST = 100.0
FOV_TEST  = 90.0
# Deliberately not a multiple of STEP_TEST, so the parity-forcing / ceil
# logic in both builders is actually exercised, not trivially satisfied.
rect = shapely_box(0.0, 0.0, 733.0, 517.0)

regression_rows = []
for fixed_axis, direction in (("y", "horizontal"), ("x", "vertical")):
    grid, xs, ys = create_grid_positions(rect, STEP_TEST, direction=direction)
    regular_path     = generate_scanning_path(grid, direction=direction)
    regular_filtered = filter_scanning_path(regular_path, rect, [], FOV_TEST)

    irregular_filtered = build_irregular_boundary_path(rect, [], STEP_TEST, FOV_TEST, fixed_axis=fixed_axis)

    shapes_match = regular_filtered.shape == irregular_filtered.shape
    exact_match  = shapes_match and bool(np.allclose(regular_filtered, irregular_filtered))
    regression_rows.append({"fixed_axis": fixed_axis, "direction": direction,
                             "n_regular": len(regular_filtered), "n_irregular": len(irregular_filtered),
                             "exact_match": exact_match})
    print(f"fixed_axis={fixed_axis} (direction={direction}): regular={len(regular_filtered)} FOVs, "
          f"irregular={len(irregular_filtered)} FOVs, exact_match={exact_match}")
    assert exact_match, (
        f"fixed_axis={fixed_axis}: irregular-grid boustrophedon path diverges from "
        f"generate_scanning_path on a plain rectangle -- CONTRADICTS the production path builder.")

print("\nPASS: irregular-grid boustrophedon path exactly reproduces generate_scanning_path "
      "on a degenerate rectangular tissue, for both fixed_axis choices.")

## Part 3 -- build the real irregular grid (LT060_sample_04, 40X)

Same real boundary/holes/canvas as `compare_60x_40x_irregular_grid_fov_coverage.ipynb`,
so results are directly comparable to that notebook's own FOV counts.

In [ ]:
BOUNDARY_DIR = MERCI_DIR / "data" / "positions" / "examples" / "lineage_tracing_mosaic"
CANVAS_PATH  = MERCI_DIR / "data" / "mosaic_canvas_examples" / "lineage_tracing" / "canvas.npz"

boundary_polygon = load_boundary_polygon(BOUNDARY_DIR / "boundary_positions.txt")
hole_polygons    = load_hole_polygons(BOUNDARY_DIR)
canvas           = load_mosaic_canvas(CANVAS_PATH)

MICROSCOPE = "ST2"
image_size_px, _  = get_camera_frame_size(MICROSCOPE)
pixel_size_60x_um = get_camera_pixel_size_um(MICROSCOPE)
pixel_size_40x_um = pixel_size_60x_um * (60.0 / 40.0)
NON_OVERLAP_FRACTION = 0.9
fov_size_um  = pixel_size_40x_um * image_size_px
step_size_um = fov_size_um * NON_OVERLAP_FRACTION
MIN_WIDTH_FRAC = 0.1

print(f"40X: fov_size={fov_size_um:.1f} um, step={step_size_um:.1f} um")

irregular_paths = {}
for fixed_axis in ("y", "x"):
    coords = build_irregular_boundary_path(boundary_polygon, hole_polygons, step_size_um, fov_size_um,
                                            fixed_axis=fixed_axis, min_width_frac=MIN_WIDTH_FRAC)
    total_um, max_step_um = get_path_stats(coords)
    irregular_paths[fixed_axis] = coords
    print(f"fixed_axis={fixed_axis}: {len(coords):4d} FOVs, path length {total_um / 1000:.2f} mm, "
          f"max single step {max_step_um:.1f} um")

## Part 4 -- does the short-return-leg parity rule matter here?

`create_grid_positions` forces the traversal axis to an EVEN count
specifically so the snake's start and end land on the SAME side (short
return leg) instead of opposite corners. `build_irregular_bands`'
`force_parity=True` applies the identical rule to the fixed axis. This
compares the real tissue's fixed-axis FOV count/parity with and without
that rule, and its effect on the actual path's own start-to-end distance
(not just the largest single step) -- the concrete case this rule is meant
to fix.

In [ ]:
parity_rows = []
for fixed_axis in ("y", "x"):
    for force_parity in (True, False):
        bands = build_irregular_bands(boundary_polygon, hole_polygons, step_size_um, fov_size_um,
                                       fixed_axis=fixed_axis, min_width_frac=MIN_WIDTH_FRAC,
                                       force_parity=force_parity)
        n_fixed = len(bands)
        path = generate_irregular_scanning_path(bands, fixed_axis=fixed_axis)
        coords = filter_scanning_path(path, boundary_polygon, hole_polygons, fov_size_um) if len(path) else path
        start_end_dist = float(np.linalg.norm(coords[-1] - coords[0])) if len(coords) > 1 else 0.0
        total_um, max_step_um = get_path_stats(coords)
        parity_rows.append({"fixed_axis": fixed_axis, "force_parity": force_parity,
                             "n_fixed_positions": n_fixed, "n_fixed_even": n_fixed % 2 == 0,
                             "n_fovs": len(coords), "start_end_distance_um": start_end_dist,
                             "max_step_um": max_step_um})

df_parity = pd.DataFrame(parity_rows)
df_parity["start_end_distance_um"] = df_parity["start_end_distance_um"].round(1)
df_parity["max_step_um"] = df_parity["max_step_um"].round(1)
df_parity

## Part 5 -- return path: does `close_scanning_path` still do the right thing?

`close_scanning_path` snaps continuous coordinates to integer grid indices
via `_grid_indices` (`(coord - min) / step_size`), then `_side_indices`
groups points by index on one axis and picks the argmin/argmax on the
other. That assumes BOTH axes sit on one shared global lattice -- true for
the regular grid, but only half-true for the irregular grid (only the
fixed axis is a shared lattice; the cross axis is independently re-centred
per band).

The natural pairing keeps this safe: request `return_side` along the CROSS
axis (`fixed_axis="y"` -> `"left"`/`"right"`, `fixed_axis="x"` ->
`"top"`/`"bottom"`) -- `_side_indices` then GROUPS by the fixed axis
(a real shared lattice, so grouping is meaningful) and only takes
argmin/argmax on the cross axis WITHIN each group (which only needs to be
correct locally, not globally aligned). Checked below against an
independent, non-quantised ground truth -- not just "it runs without
raising".

In [ ]:
def independent_side_points(coords, fixed_axis, side, step_size_um):
    """Ground-truth 'extreme point per row/column', using real coordinates
    directly (no _grid_indices quantisation) -- cross-checks
    close_scanning_path's own quantised selection.
    """
    coords = np.asarray(coords, dtype=float)
    fixed_col, cross_col = (1, 0) if fixed_axis == "y" else (0, 1)
    group_key = np.round(coords[:, fixed_col] / step_size_um).astype(int)
    selected = []
    for g in np.unique(group_key):
        idx = np.where(group_key == g)[0]
        cross = coords[idx, cross_col]
        k = idx[np.argmax(cross)] if side in ("right", "top") else idx[np.argmin(cross)]
        selected.append(int(k))
    return np.array(sorted(selected))


NATURAL_SIDES = {"y": ["left", "right"], "x": ["top", "bottom"]}

closure_rows = []
closed_paths = {}
for fixed_axis in ("y", "x"):
    coords = irregular_paths[fixed_axis]
    for side in NATURAL_SIDES[fixed_axis]:
        closed, moved_idxs = close_scanning_path(coords, step_size_um, return_side=side)

        ix, iy = _grid_indices(coords, step_size_um)
        prod_idxs  = set(_side_indices(coords, ix, iy, side).tolist()) - {0}
        truth_idxs = set(independent_side_points(coords, fixed_axis, side, step_size_um).tolist()) - {0}
        idxs_match = prod_idxs == truth_idxs

        start_preserved = bool(np.array_equal(closed[0], coords[0]))
        total_um, max_step_um = get_path_stats(closed)

        closure_rows.append({
            "fixed_axis": fixed_axis, "return_side": side,
            "n_fovs": len(coords), "n_moved": len(moved_idxs),
            "start_preserved": start_preserved, "side_idxs_match_ground_truth": idxs_match,
            "path_length_mm": total_um / 1000.0, "max_step_um": max_step_um,
        })
        closed_paths[(fixed_axis, side)] = closed
        assert start_preserved, f"fixed_axis={fixed_axis} side={side}: close_scanning_path moved the start point."
        assert idxs_match, (
            f"fixed_axis={fixed_axis} side={side}: close_scanning_path's quantised side selection "
            f"disagrees with the real-coordinate ground truth -- CONTRADICTS the return-path builder.")

df_closure = pd.DataFrame(closure_rows)
df_closure["path_length_mm"] = df_closure["path_length_mm"].round(3)
df_closure["max_step_um"] = df_closure["max_step_um"].round(1)
print("PASS: close_scanning_path's return-side selection matches the real-coordinate ground truth "
      "for every natural (fixed_axis, return_side) pairing -- start point preserved in every case.")
df_closure

## Part 5b -- the *unnatural* pairing, for contrast

For completeness: pairing the return side with the FIXED axis's own
direction instead (e.g. `fixed_axis="y"` + `return_side="top"`/`"bottom"`)
groups by the CROSS axis, which is NOT a shared lattice across bands --
`_side_indices`' quantised grouping is then expected to disagree with the
real-coordinate ground truth. Checked (not assumed) so the natural-pairing
requirement above is a demonstrated constraint, not a guess.

In [ ]:
UNNATURAL_SIDES = {"y": ["top", "bottom"], "x": ["left", "right"]}

for fixed_axis in ("y", "x"):
    coords = irregular_paths[fixed_axis]
    for side in UNNATURAL_SIDES[fixed_axis]:
        ix, iy = _grid_indices(coords, step_size_um)
        prod_idxs  = set(_side_indices(coords, ix, iy, side).tolist()) - {0}
        truth_idxs = set(independent_side_points(coords, "x" if fixed_axis == "y" else "y", side, step_size_um).tolist()) - {0}
        print(f"fixed_axis={fixed_axis} side={side} (unnatural pairing): "
              f"prod picks {len(prod_idxs)} points, ground truth (grouped by the OTHER axis) picks "
              f"{len(truth_idxs)} points, overlap={len(prod_idxs & truth_idxs)}")

## Part 6 -- automatically choosing the correct `return_side`

The diagnostic overlay in an earlier version of this notebook hardcoded
`return_side="right"` for `fixed_axis="y"` -- but Part 5's own table
already showed `"right"` gives the LARGER `max_step_um` (5762.7 um) of the
two natural sides, while `"left"` gives the smaller one (2733.8 um, equal
to the unclosed path's own max step -- i.e. closing with `"left"`
introduces no new jump at all). `"right"` was simply the wrong guess.

Rather than hardcode either guess, `determine_return_side` picks it the way
the request describes: look at which side of ITS OWN row/column the raw
(pre-closure) path's start point and end point each sit on (reusing
`_side_indices`, the same production primitive `close_scanning_path` itself
uses) -- if both agree on one side, that side is returned directly, since
the path already naturally begins/ends there (no new jump needed to close
it). If they disagree (e.g. `force_parity=False`, or a boundary shape that
breaks the same-side guarantee), it falls back to actually trying both
sides and keeping whichever gives the smaller `max_step_um` -- an
empirical tie-break, not a second guess.

In [ ]:
def determine_return_side(coords, fixed_axis, step_size_um):
    """Pick which of the two natural cross-axis sides (left/right for
    fixed_axis='y', top/bottom for fixed_axis='x') close_scanning_path
    should move to the end -- automatically, from the raw path's own
    geometry, instead of a hardcoded guess.

    Returns (side, reason) -- reason is 'start/end agree' when the direct
    geometric check decided it, or 'fallback: shorter max_step_um' when it
    had to try both sides and compare.
    """
    side_a, side_b = {"y": ("left", "right"), "x": ("top", "bottom")}[fixed_axis]
    ix, iy = _grid_indices(coords, step_size_um)
    idxs_a = set(_side_indices(coords, ix, iy, side_a).tolist())
    idxs_b = set(_side_indices(coords, ix, iy, side_b).tolist())

    def _side_of(idx):
        in_a, in_b = idx in idxs_a, idx in idxs_b
        if in_a and not in_b:
            return side_a
        if in_b and not in_a:
            return side_b
        return None   # on neither/both -- ambiguous (e.g. a single-point row/column)

    start_side, end_side = _side_of(0), _side_of(len(coords) - 1)
    if start_side is not None and start_side == end_side:
        return start_side, "start/end agree"

    best_side, best_max_step = None, None
    for side in (side_a, side_b):
        closed, _ = close_scanning_path(coords, step_size_um, return_side=side)
        _, max_step_um = get_path_stats(closed)
        if best_max_step is None or max_step_um < best_max_step:
            best_side, best_max_step = side, max_step_um
    return best_side, "fallback: shorter max_step_um"


auto_rows = []
for fixed_axis in ("y", "x"):
    coords = irregular_paths[fixed_axis]
    chosen_side, reason = determine_return_side(coords, fixed_axis, step_size_um)

    # Cross-check against Part 5's own measured max_step_um for both natural
    # sides -- the auto-picked side must never be the WORSE (larger-jump) one.
    measured = {row["return_side"]: row["max_step_um"]
                for row in closure_rows if row["fixed_axis"] == fixed_axis}
    other_side = [s for s in measured if s != chosen_side][0]
    auto_rows.append({"fixed_axis": fixed_axis, "chosen_side": chosen_side, "reason": reason,
                       "chosen_max_step_um": measured[chosen_side],
                       "other_side": other_side, "other_max_step_um": measured[other_side]})
    print(f"fixed_axis={fixed_axis}: auto-chose return_side='{chosen_side}' ({reason}) -- "
          f"max_step_um={measured[chosen_side]:.1f}, vs '{other_side}'={measured[other_side]:.1f}")
    assert measured[chosen_side] <= measured[other_side], (
        f"fixed_axis={fixed_axis}: determine_return_side picked '{chosen_side}' "
        f"(max_step={measured[chosen_side]:.1f}) but '{other_side}' is actually shorter "
        f"(max_step={measured[other_side]:.1f}) -- the auto-decision is WRONG.")

print("\nPASS: determine_return_side always picks the natural side with the smaller (or equal) "
      "return-leg max_step_um, for both fixed_axis choices, on the real benchmark.")
pd.DataFrame(auto_rows)

## Part 7 -- connectivity still holds on the final, ordered + closed path

Path ordering/closing only reorders coordinates -- it cannot change the
overlap graph -- but this re-confirms the single-connected-component hard
constraint directly on the exact array each closed-path variant above
would actually hand to the microscope, not just the raw unordered
candidate set from the earlier notebook.

In [ ]:
def connectivity_report(coords, fov_size_um, min_overlap_fraction=0.02):
    n = len(coords)
    if n == 0:
        return 0, None
    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=fov_size_um * np.sqrt(2))
    fov_area = fov_size_um ** 2
    ii, jj, frac = [], [], []
    for i, j in pairs:
        dx = abs(coords[i, 0] - coords[j, 0])
        dy = abs(coords[i, 1] - coords[j, 1])
        if dx >= fov_size_um or dy >= fov_size_um:
            continue
        frac.append((fov_size_um - dx) * (fov_size_um - dy) / fov_area)
        ii.append(i); jj.append(j)
    ii, jj, frac = np.array(ii), np.array(jj), np.array(frac)
    keep = frac >= min_overlap_fraction
    adj = coo_matrix((np.ones(keep.sum()), (ii[keep], jj[keep])), shape=(n, n))
    n_components, _ = connected_components(adj, directed=False)
    full_adj = coo_matrix((-frac, (ii, jj)), shape=(n, n))
    n_components_full, _ = connected_components(full_adj, directed=False)
    bottleneck = None
    if n_components_full == 1:
        mst = minimum_spanning_tree(full_adj)
        bottleneck = float(-mst.data.max())
    return n_components, bottleneck


for (fixed_axis, side), closed in closed_paths.items():
    n_components, bottleneck = connectivity_report(closed, fov_size_um)
    status = "OK" if n_components == 1 else "BROKEN"
    print(f"fixed_axis={fixed_axis} return_side={side:6s}: {len(closed):4d} FOVs, "
          f"components={n_components} [{status}], bottleneck overlap="
          f"{'n/a' if bottleneck is None else f'{bottleneck:.4f}'}")
    assert n_components == 1, (
        f"fixed_axis={fixed_axis} side={side}: closed path lost single-connectivity -- "
        f"reliable global alignment is not possible.")

print("\nPASS: every closed irregular-grid path is a single connected component.")

## Diagnostic overlay -- one closed path, drawn in acquisition order

Visual sanity check for the `fixed_axis="y"` variant, using
`determine_return_side` (Part 6) rather than a hardcoded guess to pick
`return_side` -- consecutive FOVs connected by a line in acquisition
order, so the boustrophedon snake shape and the short return leg (moved
points, drawn in a different colour) are both visible directly over the
real mosaic.

In [ ]:
DIAG_AXIS = "y"
DIAG_SIDE, _diag_reason = determine_return_side(irregular_paths[DIAG_AXIS], DIAG_AXIS, step_size_um)
print(f"determine_return_side picked '{DIAG_SIDE}' ({_diag_reason}) for the diagnostics below.")
diag_coords = irregular_paths[DIAG_AXIS]
diag_closed, diag_moved = close_scanning_path(diag_coords, step_size_um, return_side=DIAG_SIDE)
diag_moved_set = set(diag_moved.tolist())

covered_vals = canvas.image[canvas.covered]
vmin, vmax = np.percentile(covered_vals, [1, 99])

fig, ax = plt.subplots(figsize=(9, 11))
ax.imshow(canvas.image, cmap="gray", vmin=vmin, vmax=vmax, origin="upper")
px, py = canvas.to_px(diag_coords[:, 0], diag_coords[:, 1])
ax.plot(px, py, "-", color="yellow", lw=0.8, alpha=0.8, zorder=2, label="snake order (pre-closure)")
bx, by = boundary_polygon.exterior.xy
pbx, pby = canvas.to_px(np.asarray(bx), np.asarray(by))
ax.plot(pbx, pby, "-", color="red", lw=1.1, zorder=3, label="tissue boundary")
for hole in hole_polygons:
    hx, hy = hole.exterior.xy
    phx, phy = canvas.to_px(np.asarray(hx), np.asarray(hy))
    ax.plot(phx, phy, "--", color="cyan", lw=0.9, zorder=3)
moved_mask = np.array([i in diag_moved_set for i in range(len(diag_coords))])
ax.scatter(px[moved_mask], py[moved_mask], s=10, color="magenta", zorder=4,
           label=f"moved to end (return_side='{DIAG_SIDE}')")
ax.scatter(px[0], py[0], s=40, color="lime", marker="*", zorder=5, label="start (FOV 0)")
ax.legend(loc="upper right", fontsize=8)
ax.set_title(f"fixed_axis='{DIAG_AXIS}' irregular grid -- boustrophedon order + return_side='{DIAG_SIDE}'", fontsize=11)
ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
FIG_DIR = get_merci_figures_dir(SAMPLE_DIR, "tests", "test_irregular_grid_boustrophedon_return_path", subfolder="irregular_grid")
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "test_irregular_grid_boustrophedon_return_path_overlay.png", dpi=150, bbox_inches="tight")
plt.show()

## Diagnostic: the final path alone, no mosaic background

The overlay above is hard to read with the mosaic image underneath.  Same
`fixed_axis='y'`, `return_side='right'` case, drawn on its own in real
stage microns instead: FOV footprints as light outline squares, the path
drawn as a line colour-graded from start (dark purple) to end (bright
yellow) so the boustrophedon snake direction AND the return leg (the
sudden colour jump back across the tissue) are both immediately visible.
This is the actual POST-CLOSURE order (`diag_closed`), not the raw
pre-closure snake the overlay above shows.

In [ ]:
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

moved_coords_set = {tuple(diag_coords[i]) for i in diag_moved_set}
closed_moved_mask = np.array([tuple(p) in moved_coords_set for p in diag_closed])

fig, ax = plt.subplots(figsize=(8, 10))

half = fov_size_um / 2.0
for x, y in diag_closed:
    ax.add_patch(mpatches.Rectangle((x - half, y - half), fov_size_um, fov_size_um,
                                     lw=0.4, edgecolor="0.75", facecolor="none", zorder=1))

segments  = np.stack([diag_closed[:-1], diag_closed[1:]], axis=1)
order_idx = np.arange(len(diag_closed) - 1)
lc = LineCollection(segments, cmap="viridis", norm=Normalize(0, len(diag_closed) - 1),
                     linewidths=1.4, zorder=2)
lc.set_array(order_idx)
ax.add_collection(lc)

bx, by = boundary_polygon.exterior.xy
ax.plot(bx, by, "-", color="red", lw=1.3, zorder=3, label="tissue boundary")
for hole in hole_polygons:
    hx, hy = hole.exterior.xy
    ax.plot(hx, hy, "--", color="cyan", lw=1.0, zorder=3)

ax.scatter(diag_closed[closed_moved_mask, 0], diag_closed[closed_moved_mask, 1],
           s=20, color="magenta", zorder=4, label=f"moved to end (return_side='{DIAG_SIDE}')")
ax.scatter(*diag_closed[0],  s=100, color="lime",  marker="*", edgecolor="black", linewidth=0.5,
           zorder=5, label="start (FOV 0)")
ax.scatter(*diag_closed[-1], s=100, color="black", marker="X", zorder=5, label="end (last FOV)")

cbar = fig.colorbar(lc, ax=ax, fraction=0.04, pad=0.02)
cbar.set_label("acquisition order", fontsize=9)

ax.set_xlabel("Stage X (um)", fontsize=10)
ax.set_ylabel("Stage Y (um)", fontsize=10)
ax.set_title(f"fixed_axis='{DIAG_AXIS}' irregular grid -- path only, no mosaic\n"
             f"{len(diag_closed)} FOVs, return_side='{DIAG_SIDE}'", fontsize=11)
ax.set_aspect("equal")
ax.invert_yaxis()   # match the mosaic overlay's orientation above (stage y increases downward there)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "test_irregular_grid_boustrophedon_return_path_path_only.png", dpi=150, bbox_inches="tight")
plt.show()

## Takeaways

- **Boustrophedon path -- no contradiction**: `generate_irregular_scanning_path`
  exactly reproduces production's `generate_scanning_path` on a degenerate
  rectangular tissue for both `fixed_axis` choices (`np.allclose`, not
  eyeballed) -- see Part 2. The generalisation is safe.
- **Return path -- no contradiction, PROVIDED the natural axis pairing is
  used**: `close_scanning_path`'s quantised `_side_indices` selection
  matches an independent, real-coordinate ground truth exactly when
  `return_side` is requested along the CROSS axis (`fixed_axis="y"` with
  `"left"`/`"right"`, `fixed_axis="x"` with `"top"`/`"bottom"`) -- see
  Part 5. The reverse pairing (Part 5b) is NOT expected to agree (the
  cross axis is not a shared lattice across bands) -- any future
  production wrapper for this method must enforce/hardcode the natural
  pairing rather than accepting an arbitrary `return_side`.
- **`return_side` must be chosen, not guessed -- an earlier version of this
  very notebook guessed wrong**: the first diagnostic overlay hardcoded
  `fixed_axis="y"` + `return_side="right"`, which Part 5's own table shows
  is the WORSE of the two natural sides (`max_step_um` 5762.7 vs 2733.8 for
  `"left"`) -- a real, human-made mistake this notebook caught on itself.
  `determine_return_side` (Part 6) fixes this properly: it checks which
  side of its own row/column the raw path's start point and end point each
  sit on (reusing `_side_indices`, the same primitive `close_scanning_path`
  itself uses), and picks that side directly when they agree -- falling
  back to trying both and comparing `max_step_um` when they don't. Verified
  against Part 5's own measurements: the auto-picked side is never the
  worse one, for both `fixed_axis` choices, on the real benchmark (`assert`,
  Part 6). The diagnostic images below now use this function instead of a
  hardcoded guess.
- **Parity rule matters and needed adding**: Method 2's original prototype
  never forced the fixed axis to an even FOV count the way
  `create_grid_positions` forces its own traversal axis -- Part 4's table
  shows the actual effect (`force_parity=True` vs `False`) on this real
  tissue's fixed-axis FOV count/parity and the path's real start-to-end
  distance. `force_parity=True` (added here, reusing production's own
  `spaced_coords`) is the correct default for a production version, for
  the identical reason `create_grid_positions` already forces it. It is
  also a precondition for `determine_return_side`'s direct
  (non-fallback) path: the same-side guarantee it relies on only holds
  when the fixed axis has an even count.
- **Connectivity preserved**: every closed path (both axis choices, both
  natural return sides) remains a single connected component -- Part 7.
- **The WRONG return_side can still be a large single step -- exactly why
  auto-detection matters**: `fixed_axis="y"`, `return_side="right"`
  produced the biggest `max_step_um` of any variant tested (5762.8 um, see
  Part 5's table) -- the tissue's right edge is notched/protruded, so
  consecutive rightmost points across different rows are genuinely far
  apart in real space even though they're adjacent once moved to the end
  of the path. `close_scanning_path` never reorders the moved segment
  beyond a simple reversal (true for the regular grid too, and true of
  the CORRECT side as well in general) -- this is an existing, unchanged
  limitation of the return-leg step, not something the irregular grid
  introduces, but it is a good reason to always pick the side that avoids
  triggering it when a shorter option exists.
- **Net conclusion**: keeping one axis fixed/regular and letting the other
  be irregular does NOT contradict the boustrophedon-path or return-path
  logic already in `acquisition/positions.py`, as long as (a) the fixed
  axis reuses `spaced_coords`' own even-parity rule, (b) any `return_side`
  a caller requests is restricted to the cross axis, and (c) that side is
  chosen via `determine_return_side` rather than guessed. Safe to promote
  into `acquisition/positions.py` as a new option alongside
  `create_grid_positions`/`generate_scanning_path`/`close_scanning_path` --
  not done in this notebook, per the request to test first.